In [3]:
import neworder as no
import pandas as pd
import numpy as np

# --- Configuration ---
# Define the start and end years for the simulation
START_YEAR = 2025
END_YEAR = 2035
# Define the initial population size for the simulation
INITIAL_POPULATION = 100000

# --- Demographic Rates (Simplified for demonstration) ---
# These rates are illustrative and not based on actual New Zealand data.
# For a real-world simulation, these would be derived from detailed demographic statistics.

# Age-specific death rates (per 1000 people per year)
# Keys represent the start of an age band
DEATH_RATES = {
    0: 0.005,  # 0-4 years
    5: 0.0001,
    15: 0.0002,
    25: 0.0003,
    35: 0.0004,
    45: 0.0007,
    55: 0.001,
    65: 0.005,
    75: 0.01,
    85: 0.05,
    95: 0.15,
    100: 0.30
}

# Age-specific fertility rates (births per woman per year)
# Assuming only females aged 15-49 can give birth
FERTILITY_RATES = {
    15: 0.01,
    20: 0.05,
    25: 0.10,
    30: 0.12,
    35: 0.08,
    40: 0.03,
    45: 0.005
}

# Sex ratio at birth (proportion of male births)
SEX_RATIO_AT_BIRTH = 0.51

# --- Agent-based Model Definition ---
class NZPopulationAgentModel:
    """
    A component of the neworder model, representing the population dynamics.
    This class manages the agents (individuals) and their demographic processes.
    """
    def __init__(self):
        """
        Initializes the population dataframe.
        """
        # Initialize the population as a pandas DataFrame
        # Each row represents an individual (agent)
        # 'id': Unique identifier for each individual
        # 'age': Age of the individual in years
        # 'sex': 0 for female, 1 for male
        self.population = pd.DataFrame({
            "id": np.arange(INITIAL_POPULATION),
            "age": np.random.rand(INITIAL_POPULATION) * 90, # Random initial age up to 90, using numpy's rand
            "sex": (np.random.rand(INITIAL_POPULATION) < 0.5).astype(int) # Random sex distribution (0.5 for female/male), using numpy's rand
        })

    def _get_rate_for_age(self, age, rates_dict):
        """
        Helper function to get the appropriate rate for a given age from a dictionary of age bands.
        """
        age_band = 0
        for band_start in sorted(rates_dict.keys()):
            if age >= band_start:
                age_band = band_start
            else:
                break
        return rates_dict.get(age_band, 0.0)

    def _age_population(self):
        """
        Increments the age of all individuals by one year.
        """
        self.population["age"] += 1

    def _apply_mortality(self):
        """
        Applies mortality based on age-specific death rates.
        Individuals die if a random number is less than their age-specific death probability.
        """
        # Calculate individual probabilities of death
        death_probabilities = self.population["age"].apply(
            lambda age: self._get_rate_for_age(age, DEATH_RATES)
        )

        # Determine who dies based on random numbers
        will_die = np.random.rand(len(self.population)) < death_probabilities
        
        self.population = self.population[~will_die] # Remove individuals who die

    def _apply_fertility(self):
        """
        Applies fertility to female individuals within childbearing age.
        New individuals (babies) are added to the population.
        """
        # Select fertile females (arbitrarily 15-49 for this example)
        fertile_females = self.population[
            (self.population["sex"] == 0) & # Female
            (self.population["age"] >= 15) &
            (self.population["age"] <= 49)
        ]

        if fertile_females.empty:
            return

        # Calculate individual probabilities of giving birth
        birth_probabilities = fertile_females["age"].apply(
            lambda age: self._get_rate_for_age(age, FERTILITY_RATES)
        )

        # Determine which females give birth
        # Multiply by 0.1 to scale down the probability for a single year step if rates are annual
        will_give_birth = np.random.rand(len(fertile_females)) < birth_probabilities * 0.01
        
        new_births = will_give_birth.sum()

        if new_births > 0:
            # Generate new babies
            new_ids = np.arange(self.population["id"].max() + 1, self.population["id"].max() + 1 + new_births)
            new_ages = np.zeros(new_births) # Babies start at age 0
            new_sexes = (np.random.rand(new_births) < SEX_RATIO_AT_BIRTH).astype(int) # Assign sex based on ratio

            new_individuals = pd.DataFrame({
                "id": new_ids,
                "age": new_ages,
                "sex": new_sexes
            })

            self.population = pd.concat([self.population, new_individuals], ignore_index=True)

# --- neworder Model Class ---
class NewZealandPopulationSimulation(no.Model):
    """
    Encapsulates the entire neworder simulation model for the New Zealand population.
    """
    def __init__(self):
        """
        Initializes the model with its timeline, systems, and checkpoint callback.
        """
        # Pass a range of years as the timeline to the base Model class constructor.
        # This implicitly sets the timeline start, end, and step.
        super().__init__(range(START_YEAR, END_YEAR + 1))

        # Define the systems as an attribute of the model instance.
        # These systems will be managed by the neworder runtime.
        self.systems = {
            "nz_population_agent_model": NZPopulationAgentModel()
        }

        no.log("Initialising NewZealandPopulationSimulation model...")
        no.log(f"Initial population size: {len(self.systems['nz_population_agent_model'].population)}")
        no.log(f"Initial population age distribution:\n{self.systems['nz_population_agent_model'].population['age'].describe()}")
        no.log(f"Initial population sex distribution:\n{self.systems['nz_population_agent_model'].population['sex'].value_counts()}")

    def step(self):
        """
        Defines the operations to perform at each simulation step (e.g., each year).
        This method will be called by the neworder runtime for each timestep.
        """
        # Access the timeline index through the model's own timeline attribute
        current_year = self.timeline.index()
        no.log(f"Simulation year: {current_year}")

        # Access the agent model and apply demographic processes
        self.systems["nz_population_agent_model"]._age_population()
        self.systems["nz_population_agent_model"]._apply_mortality()
        self.systems["nz_population_agent_model"]._apply_fertility()

        # Check for extinction (optional, but good for stability)
        if len(self.systems["nz_population_agent_model"].population) == 0:
            no.log("Population extinct. Halting simulation.")
            return False # Halts the simulation

        # Log population changes for the current step
        population_size = len(self.systems["nz_population_agent_model"].population)
        no.log(f"Population size at end of year {current_year}: {population_size}")

        return True # Continue the simulation

    def checkpoint(self):
        """
        Captures key metrics at each checkpoint and prints them.
        This method will be called by the neworder runtime at each checkpoint.
        """
        # Access the timeline index through the model's own timeline attribute
        current_year = self.timeline.index()
        population_size = len(self.systems["nz_population_agent_model"].population)
        average_age = self.systems["nz_population_agent_model"].population["age"].mean() if population_size > 0 else 0

        no.log(f"Checkpoint at year {current_year}: Population size = {population_size}, Average age = {average_age:.2f}")

    def finalise(self):
        """
        Method called at the end of the simulation.
        Can be used for final data aggregation or output.
        """
        no.log("Simulation finished.")
        final_population = self.systems["nz_population_agent_model"].population
        no.log(f"Final population size: {len(final_population)}")
        no.log(f"Final population age distribution:\n{final_population['age'].describe()}")
        no.log(f"Final population sex distribution:\n{final_population['sex'].value_counts()}")

# --- Run the simulation ---
# Create an instance of the NewZealandPopulationSimulation model and run it.
model = NewZealandPopulationSimulation()
no.run(model)


<frozen importlib._bootstrap>:488: RuntimeWarning: neworder installed in serial mode. If necessary, enable MPI with: pip install neworder[parallel]


TypeError: __init__(): incompatible constructor arguments. The following argument types are supported:
    1. _neworder_core.Model(timeline: _neworder_core.Timeline, seeder: Callable = <built-in method of PyCapsule object at 0x11a8b9350>)

Invoked with: range(2025, 2036)

In [2]:
!pip install neworder

  Using cached neworder-1.4.2.tar.gz (37 kB)
  Installing build dependencies ... one
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pandas_stubs-2.2.3.250527-py3-none-any.whl.metadata (10 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached types_pytz-2025.2.0.20250516-py3-none-any.whl.metadata (1.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 4.5 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 4.5 MB/s eta 0:00:0000:0100:01
Using cached pandas_stubs-2.2.3.250527-py3-none-any.whl (159 kB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached types_pytz-2025.2.0.20250516-py3-none-any.whl (10 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
  Created wheel for neworder: filename=neworder-1.4.2-cp313-cp313-macosx_11_0_arm64.whl size=208637 sha256=e6f1180

In [9]:

#!/usr/bin/env python3
"""
Open-Source IDMOD Microsimulation adapted to Jupyter/Neworder usage
Fully parameterised inline; includes:
  - Familial & non-familial ID cohorts
  - Ageing, mortality, fertility
  - Diagnostic odyssey vs WGS
  - Cost and QALY accumulation
  - Checkpoint and finalise methods for neworder compatibility
Run in Jupyter or script with:
    pip install neworder pandas numpy matplotlib
    python idmod_neworder_full.py
"""
# neworder removed for manual loop
# import neworder as no
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- 1. CONFIGURATION & PARAMETERS -----------------------------------------
START_YEAR = 2020
END_YEAR   = 2100
POP_SIZE   = 62000
DISCOUNT   = 0.035

# Proportions
FAMILIAL_PROP = 0.27
GENETIC_PROP  = 0.45
SEVERITY_DIST = {"mild":0.70, "moderate":0.30}

# Age bands & distribution
AGE_BREAKS = np.arange(0,95,5)
AGE_DIST   = np.array([
    0.064,0.067,0.069,0.069,0.071,0.076,0.080,0.077,0.080,
    0.079,0.073,0.078,0.077,0.072,0.055,0.041,0.031,0.022,0.012
])
AGE_DIST /= AGE_DIST.sum()

def sample_ages(n):
    idx = np.random.choice(len(AGE_BREAKS), size=n, p=AGE_DIST)
    return AGE_BREAKS[idx] + np.random.randint(0,5,size=n)

# Fertility rates per severity
FERTILITY = {"mild":0.015, "moderate":0.005}

# Mortality
ABS_MORT = [
    0.0002,0.0001,0.0001,0.0002,0.0003,0.0005,0.0007,
    0.0011,0.0019,0.0035,0.0055,0.0085,0.0130,
    0.0200,0.0350,0.0600,0.0950,0.1500,0.2500
]
MORT_HR = {"mild":2.0, "moderate":4.0}

# Diagnostic
YIELD_STD = 0.25
YIELD_WGS = 0.45
COST_STD  = 5000
COST_WGS  = 15000

# Costs
COST_LIVING = {"residential":16973, "sda":(4452+59375)/2, "sil":1727*52, "home":0}
RESPITE     = 13024
EDU_COST    = {
    ("primary","supp"):16107,("primary","sub"):27903,("primary","ext"):46733,
    ("secondary","supp"):18958,("secondary","sub"):30789,("secondary","ext"):49604
}
EI_COST_HR = 185
EI_HRS_WK  = 5
TRANS_DSP  = {"u18_home":8104,"u20_indep":13335,"18_20_home":9393,"20_plus":23254}
MOBILITY   = 2493
EMP_LOSS   = {"none":0,"supported":10000,"open":0}

# Utilities
U_BASE    = {"mild":0.78,"moderate":0.65}
U_DIAG_BOOST = 0.02
U_CARER   = 0.05

# --- 2. SYNTHETIC POPULATION GENERATOR -------------------------------------
def gen_population(n):
    df = pd.DataFrame({
        'age': sample_ages(n),
        'sex': np.random.choice(['F','M'],size=n),
        'severity': np.random.choice(list(SEVERITY_DIST),size=n,p=list(SEVERITY_DIST.values())),
        'familial': np.random.rand(n)<FAMILIAL_PROP,
        'living': np.random.choice(list(COST_LIVING),size=n,p=[0.2,0.1,0.3,0.4]),
        'school': np.where(sample_ages(n)<5,None,np.where(sample_ages(n)<12,'primary','secondary')),
        'support': np.random.choice(['supp','sub','ext'],size=n,p=[0.5,0.3,0.2]),
        'uses_respite':np.random.rand(n)<0.3,
        'needs_mobility':np.random.rand(n)<0.2,
        'employment':np.random.choice(['none','supported','open'],size=n,p=[0.7,0.2,0.1]),
        'diagnosed':False
    })
    df['annual_cost']=0.0
    df['annual_qaly']=0.0
    return df

# --- 3. POPULATION SYSTEM ----------------------------------------------------
class PopSystem:
    def __init__(self):
        self.population = gen_population(int(POP_SIZE))
    def age_and_mortality(self):
        df=self.population; df['age']+=1
        rates=np.array([ABS_MORT[min(a//5,len(ABS_MORT)-1)] for a in df['age']])
        hr   = df['severity'].map(MORT_HR).values
        keep = np.random.rand(len(df)) > (rates*hr)
        self.population=df.loc[keep].reset_index(drop=True)
    def fertility(self):
        df=self.population
        mask=(df['sex']=='F') & df['age'].between(15,45)
        rates=df.loc[mask,'severity'].map(FERTILITY).values
        births=np.random.rand(mask.sum())<rates
        n=births.sum()
        if n>0:
            babies=gen_population(int(n)); babies['age']=0
            moms=df.loc[mask].reset_index(); babies['familial']=moms.loc[births,'familial'].values
            self.population=pd.concat([df,babies],ignore_index=True)
    def diagnosis(self):
        df=self.population
        mask=df['familial'] & ~df['diagnosed']
        df.loc[mask,'diagnosed']=np.random.rand(mask.sum())<YIELD_WGS
        self.population=df
    def cost_qaly(self):
        df=self.population; df['annual_cost']=0; df['annual_qaly']=0
        for lv,c in COST_LIVING.items(): df.loc[df['living']==lv,'annual_cost']+=c
        df.loc[df['uses_respite'],'annual_cost']+=RESPITE
        sch=df['school'].notna()
        df.loc[sch,'annual_cost']+=df.loc[sch].apply(lambda r:EDU_COST[(r['school'],r['support'])],axis=1)
        ei=df['age']<5; df.loc[ei,'annual_cost']+=EI_HRS_WK*52*EI_COST_HR
        df.loc[df['needs_mobility'],'annual_cost']-=MOBILITY
        df['annual_cost']+=df['employment'].map(EMP_LOSS)
        df['annual_qaly']=df['severity'].map(U_BASE)+np.where(df['diagnosed'],U_DIAG_BOOST,0)+U_CARER
        self.population=df
    def step(self):
        self.age_and_mortality(); self.fertility(); self.diagnosis(); self.cost_qaly()

# --- 4. TIMELINE & MODEL ------------------------------------------------------
class SimTimeline(no.Timeline):
    def at_start(self): return self.index()==START_YEAR
    def at_end(self):   return self.index()>=END_YEAR

class IDMODModel(no.Model):
    def __init__(self):
        super().__init__(SimTimeline())
        self.systems={'pop':PopSystem()}
        self.cost_accum=0.0
        self.qaly_accum=0.0
    def step(self):
        yr=self.timeline.index(); self.systems['pop'].step()
        disc=1/((1+DISCOUNT)**(yr-START_YEAR))
        df=self.systems['pop'].population
        self.cost_accum+=df['annual_cost'].sum()*disc
        self.qaly_accum+=df['annual_qaly'].sum()*disc
        return not self.timeline.at_end()
    def checkpoint(self): pass
    def finalise(self):
        no.log(f"Total discounted cost: ${self.cost_accum:,.0f}")
        no.log(f"Total discounted QALYs: {self.qaly_accum:,.1f}")

# --- 5. RUN & SIMPLE ANALYSES -----------------------------------------------
if __name__=="__main__":
    model=IDMODModel(); no.run(model)
    # Example downstream plot of cohort size over time omitted for brevity


RuntimeError: Unable to cast Python instance of type <class 'method'> to C++ type '?' (#define PYBIND11_DETAILED_ERROR_MESSAGES or compile in debug mode for details)